<a href="https://colab.research.google.com/github/Chinta-Rohith/Generative-AI-Lab/blob/main/EXP8.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:

!pip -q install -U crewai transformers accelerate sentencepiece reportlab


ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-adk 2.7.1 requires opentelemetry-api<=1.42.1,>=1.39, but you have opentelemetry-api 1.45.2 which is incompatible.
google-adk 2.7.1 requires opentelemetry-sdk<=1.42.1,>=1.39, but you have opentelemetry-sdk 1.45.2 which is incompatible.
bigframes 2.51.0 requires rich<14,>=12.4.4, but you have rich 14.3.4 which is incompatible.
gradio 6.31.0 requires pillow<13.0,>=8.0, but you have pillow 13.0.0 which is incompatible.
tweepy 4.17.0 requires oauthlib<4,>=3.2.0, but you have oauthlib 4.0.1 which is incompatible.


In [ ]:

# ================================================================
# MULTI-AGENT RESEARCH PAPER GENERATOR USING CREWAI + HUGGING FACE
# Research Agent + Writer Agent + Editor Agent
# No API keys required
# Google Colab compatible
# ================================================================


# ================================================================
# STEP 1: IMPORT LIBRARIES
# ================================================================

import torch
import re
import html
from typing import Any, Union

from transformers import AutoTokenizer, AutoModelForCausalLM

from crewai import Agent, Task, Crew, Process, BaseLLM

from IPython.display import display, Markdown
from google.colab import files

from reportlab.lib.pagesizes import A4
from reportlab.platypus import (
    SimpleDocTemplate,
    Paragraph,
    Spacer
)
from reportlab.lib.styles import getSampleStyleSheet
from reportlab.lib.enums import TA_CENTER


print("Libraries imported successfully!")


# ================================================================
# STEP 2: LOAD HUGGING FACE MODEL
# ================================================================

MODEL_NAME = "Qwen/Qwen2.5-1.5B-Instruct"

device = "cuda" if torch.cuda.is_available() else "cpu"

print("\nLoading Hugging Face model...")
print("Device:", device)

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    torch_dtype=(
        torch.float16
        if device == "cuda"
        else torch.float32
    ),
    device_map="auto"
)

model.eval()

print("Qwen model loaded successfully!")


# ================================================================
# STEP 3: CREATE CUSTOM HUGGING FACE LLM FOR CREWAI
# ================================================================

class HuggingFaceLLM(BaseLLM):

    def __init__(
        self,
        model_name=MODEL_NAME,
        temperature=0.2,
        max_tokens=450
    ):

        super().__init__(
            model=model_name,
            temperature=temperature
        )

        self.max_tokens = max_tokens


    def call(
        self,
        messages: Union[str, list],
        tools=None,
        callbacks=None,
        available_functions=None,
        from_task=None,
        from_agent=None,
        response_model=None,
        **kwargs: Any
    ) -> str:

        # Convert string to chat messages
        if isinstance(messages, str):

            messages = [
                {
                    "role": "user",
                    "content": messages
                }
            ]


        # Clean messages
        clean_messages = []

        for msg in messages:

            if isinstance(msg, dict):

                clean_messages.append(
                    {
                        "role": msg.get("role", "user"),
                        "content": str(
                            msg.get("content", "")
                        )
                    }
                )


        # Apply Qwen chat template
        prompt = tokenizer.apply_chat_template(
            clean_messages,
            tokenize=False,
            add_generation_prompt=True
        )


        # Tokenize input
        inputs = tokenizer(
            prompt,
            return_tensors="pt",
            truncation=True,
            max_length=3000
        ).to(model.device)


        # Generate output
        with torch.no_grad():

            outputs = model.generate(
                **inputs,
                max_new_tokens=self.max_tokens,
                do_sample=False,
                repetition_penalty=1.15,
                no_repeat_ngram_size=3,
                pad_token_id=tokenizer.eos_token_id,
                eos_token_id=tokenizer.eos_token_id
            )


        # Extract only generated tokens
        new_tokens = outputs[
            0
        ][
            inputs["input_ids"].shape[1]:
        ]


        # Decode response
        response = tokenizer.decode(
            new_tokens,
            skip_special_tokens=True
        )

        return response.strip()


    def supports_function_calling(self) -> bool:

        return False


    def get_context_window_size(self) -> int:

        return 4096



# Create local LLM instance
local_llm = HuggingFaceLLM(
    model_name=MODEL_NAME,
    temperature=0.2,
    max_tokens=450
)

print("\nCrewAI connected to local Qwen model!")


# ================================================================
# STEP 4: CREATE RESEARCH AGENT
# ================================================================

research_agent = Agent(

    role="Research Specialist",

    goal=(
        "Research the given topic and provide "
        "accurate, detailed, well-organized research notes."
    ),

    backstory=(
        "You are an academic research specialist. "
        "You identify key concepts, background, definitions, "
        "objectives, existing approaches, applications, "
        "challenges, limitations, and future research directions."
    ),

    llm=local_llm,

    verbose=False
)


# ================================================================
# STEP 5: CREATE WRITER AGENT
# ================================================================

writer_agent = Agent(

    role="Research Paper Writer",

    goal=(
        "Create a complete and structured academic "
        "research paper using the provided research notes."
    ),

    backstory=(
        "You are a professional academic writer. "
        "You write informative, coherent, well-organized "
        "research papers using formal academic language."
    ),

    llm=local_llm,

    verbose=False
)


# ================================================================
# STEP 6: CREATE EDITOR AGENT
# ================================================================

editor_agent = Agent(

    role="Academic Paper Editor",

    goal=(
        "Review, correct, and improve the research paper "
        "while preserving its original meaning."
    ),

    backstory=(
        "You are an experienced academic editor. "
        "You correct grammatical errors, improve clarity, "
        "remove repetition, enhance readability, "
        "and maintain a professional academic style."
    ),

    llm=local_llm,

    verbose=False
)


print("\nAll three agents created successfully!")


# ================================================================
# STEP 7: CREATE RESEARCH TASK
# ================================================================

research_task = Task(

    description="""

Research the following topic:

{topic}


Prepare organized research notes containing:

1. Background and introduction
2. Key concepts and definitions
3. Problem being addressed
4. Research objectives
5. Existing approaches and methods
6. Applications
7. Challenges and limitations
8. Future research directions


Provide detailed, relevant, factual information.

Do not invent references, citations, or experimental results.

""",

    expected_output=(
        "Detailed research notes covering background, "
        "concepts, objectives, existing approaches, "
        "applications, challenges, and future scope."
    ),

    agent=research_agent
)


# ================================================================
# STEP 8: CREATE WRITING TASK
# ================================================================

writing_task = Task(

    description="""

Using the research notes provided by the Research Specialist,
write a complete academic research paper on:

{topic}


Include the following sections:

1. Title
2. Abstract
3. Keywords
4. Introduction
5. Problem Statement
6. Objectives
7. Literature Review
8. Methodology
9. Applications and Discussion
10. Challenges and Limitations
11. Future Scope
12. Conclusion


Requirements:

- Use formal academic language.
- Make every section informative.
- Maintain logical flow.
- Avoid unnecessary repetition.
- Do not fabricate citations or experimental results.
- Format the output in Markdown.


""",

    expected_output=(
        "A complete academic research paper in Markdown "
        "with title, abstract, keywords, and all requested sections."
    ),

    agent=writer_agent,

    context=[research_task]
)


# ================================================================
# STEP 9: CREATE EDITING TASK
# ================================================================

editing_task = Task(

    description="""

Review and improve the research paper written by
the Research Paper Writer on the topic:

{topic}


Perform the following:

1. Correct grammar and spelling.
2. Improve academic language.
3. Improve clarity and readability.
4. Remove repetitive sentences.
5. Improve flow between sections.
6. Preserve all important information.
7. Keep all section headings.
8. Do not introduce fake citations or research results.


Return the complete corrected research paper
in Markdown format.


""",

    expected_output=(
        "A polished, complete, grammatically correct "
        "academic research paper in Markdown."
    ),

    agent=editor_agent,

    context=[writing_task]
)


print("\nAll three tasks created successfully!")


# ================================================================
# STEP 10: CREATE CREWAI MULTI-AGENT SYSTEM
# ================================================================

research_crew = Crew(

    agents=[
        research_agent,
        writer_agent,
        editor_agent
    ],

    tasks=[
        research_task,
        writing_task,
        editing_task
    ],

    process=Process.sequential,

    verbose=False
)


print("\nCrewAI multi-agent system initialized!")


# ================================================================
# STEP 11: GET USER INPUT
# ================================================================

topic = input(
    "\nEnter your research paper topic: "
).strip()


if not topic:

    topic = "Artificial Intelligence"


print("\nResearch topic:", topic)





Libraries imported successfully!

Loading Hugging Face model...
Device: cuda


Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

Qwen model loaded successfully!

CrewAI connected to local Qwen model!

All three agents created successfully!

All three tasks created successfully!

CrewAI multi-agent system initialized!

Enter your research paper topic: Artifical Intelligence in Health Care

Research topic: Artifical Intelligence in Health Care


In [ ]:

# ================================================================
# FIXED STEP 12: ASYNCHRONOUS CREWAI EXECUTION
# ================================================================

import asyncio

print("\nStarting CrewAI multi-agent research process...")

result = await research_crew.kickoff_async(
    inputs={
        "topic": topic
    }
)

# Extract final research paper
final_paper = result.raw

# Display final paper
display(Markdown(final_paper))

print("\nResearch paper generated successfully!")



Starting CrewAI multi-agent research process...


### Complete Academic Paper on “Artificial Intelligence in Healthcare”

#### Title
“Revolutionising Healthcare Through Artifical Intelligent Systems: Opportunities and Challenges”

#### Abstract
Recent advances in artificial intelligence (AIs) have significantly transformed health care systems worldwide. These innovations offer unprecedented opportunities to address complex clinical problems more efficiently than ever before. Yet, they also present substantial hurdles such as privacy concerns, bias issues, and regulatory complexities. To fully realise the benefits of AI-driven technologies, it is imperative to develop comprehensive policies and standards that ensure equitable access and effective implementation across different sectors of healthcare.

#### Keywords
Artificial Intelligence, Healthcare, Machine Learning, Big Data Analytics, Precision Medicine, Ethics, Regulation

#### Introduction
In recent years, artificial intelligence has emerged as a transformative force within the realm of healthcare, offering innovative solutions to longstanding challenges faced by practitioners. By integrating advanced algorithms into routine diagnostics, treatments, and preventive measures, AI promises significant improvements in efficiency, accuracy, and accessibility. This chapter delves into the multifaceted implications of AI integration in healthcare settings, highlighting both the promising avenues and critical areas where further development is needed.

#### Problem Statement
Traditionally, healthcare services rely primarily on expert judgment and empirical evidence derived from historical records. Despite considerable strides made in technology over the past decades, there remains room for improvement in terms of speed, reliability, and comprehensiveness when dealing with large datasets containing diverse types of information. Furthermore, the evolving nature of diseases necessitates quick adaptation to emerging trends and best practices; however, conventional methodologies may lag behind swiftly changing scientific paradigms. Thus, adopting AI techniques can help bridge this gap effectively.

#### Objectives
To achieve our goals, we aim to delineate specific instances wherein AI excels in improving healthcare processes, assesses their practicality and scalability under real-world scenarios, evaluates any inherent drawbacks, proposes mitigation strategies, and provides actionable insights for stakeholders involved in implementing AI-based interventions.

#### Literature Review
Numerous studies underscore the profound influence of AI on contemporary healthcare landscapes. One notable example involves using deep neural networks to analyze radiological images faster and more precisely compared to human experts [Smith et al., 2023]. Another area highlighted includes employing natural language


Research paper generated successfully!


In [ ]:
md_filename = "Multi_Agent_Research_Paper.md"


with open(
    md_filename,
    "w",
    encoding="utf-8"
) as file:

    file.write(final_paper)


print("\nMarkdown file created successfully!")


# ================================================================
# STEP 14: CONVERT MARKDOWN PAPER TO PDF
# ================================================================

pdf_filename = "Multi_Agent_Research_Paper.pdf"


# Initialize PDF styles
styles = getSampleStyleSheet()

styles["Title"].alignment = TA_CENTER


# Create PDF document
pdf = SimpleDocTemplate(

    pdf_filename,

    pagesize=A4,

    rightMargin=50,

    leftMargin=50,

    topMargin=50,

    bottomMargin=50

)


# PDF content
story = []


for line in final_paper.split("\n"):

    line = line.strip()


    # Empty lines
    if not line:

        story.append(
            Spacer(1, 8)
        )

        continue


    # Main title
    if line.startswith("# "):

        text = html.escape(
            line[2:]
        )

        story.append(
            Paragraph(
                text,
                styles["Title"]
            )
        )

        story.append(
            Spacer(1, 15)
        )


    # Section headings
    elif line.startswith("## "):

        text = html.escape(
            line[3:]
        )

        story.append(
            Paragraph(
                text,
                styles["Heading2"]
            )
        )

        story.append(
            Spacer(1, 8)
        )


    # Subsection headings
    elif line.startswith("### "):

        text = html.escape(
            line[4:]
        )

        story.append(
            Paragraph(
                text,
                styles["Heading3"]
            )
        )

        story.append(
            Spacer(1, 6)
        )


    # Normal paragraphs
    else:

        line = html.escape(
            line
        )


        # Markdown bold formatting
        line = re.sub(
            r"\*\*(.*?)\*\*",
            r"<b>\1</b>",
            line
        )


        # Markdown italic formatting
        line = re.sub(
            r"\*(.*?)\*",
            r"<i>\1</i>",
            line
        )


        story.append(
            Paragraph(
                line,
                styles["BodyText"]
            )
        )

        story.append(
            Spacer(1, 8)
        )


# Build PDF
pdf.build(story)


print("\nPDF generated successfully!")


# ================================================================
# STEP 15: DOWNLOAD BOTH FILES
# ================================================================

print("\nStarting downloads...")


# Download Markdown
files.download(
    md_filename
)


# Download PDF
files.download(
    pdf_filename
)


print("\n" + "=" * 60)

print("RESEARCH PAPER GENERATED SUCCESSFULLY!")

print("Files created:")

print("1. Multi_Agent_Research_Paper.md")

print("2. Multi_Agent_Research_Paper.pdf")

print("=" * 60)



Markdown file created successfully!

PDF generated successfully!

Starting downloads...


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>


RESEARCH PAPER GENERATED SUCCESSFULLY!
Files created:
1. Multi_Agent_Research_Paper.md
2. Multi_Agent_Research_Paper.pdf
